# CAER-Net Reproduction on CAER-S

Notebook utama untuk reproduksi CAER-Net-S pada layout repo saat ini.


In [ ]:
import os
import csv
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from tqdm import tqdm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    wandb = None
    WANDB_AVAILABLE = False


In [ ]:
WORKDIR = Path.cwd().resolve()
DATASET_DIR = WORKDIR / "CAER-S"
DETECTOR_DIR = WORKDIR / "detectors"
MANIFEST_PATH = WORKDIR / "caers_manifest.jsonl"
CKPT_ROOT = WORKDIR / "checkpoints"

CFG = {
    "project_name": "caer-net-reproduction",
    "run_name": os.environ.get("RUN_NAME", f"caernet_s_{datetime.now().strftime('%Y%m%d_%H%M%S')}"),
    "seed": 42,
    "epochs": 60,
    "per_gpu_batch_size": 32,
    "lr": 5e-3,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "scheduler_step_size": 4,
    "scheduler_gamma": 0.1,
    "num_workers_per_gpu": 2,
    "use_amp": True,
    "grad_clip_max_norm": 5.0,
    "early_stopping_patience": 10,
    "early_stopping_enabled": True,
    "wandb_mode": os.environ.get("WANDB_MODE", "online" if os.environ.get("WANDB_API_KEY") else "offline"),
    "wandb_entity": os.environ.get("WANDB_ENTITY", ""),
    "resume_from": "",  # contoh: "checkpoints/<run_name>/last.pt"
}

RUN_NAME = CFG["run_name"]
CKPT_DIR = CKPT_ROOT / RUN_NAME
BEST_CKPT_PATH = CKPT_DIR / "best.pt"
LAST_CKPT_PATH = CKPT_DIR / "last.pt"
CONFIG_PATH = CKPT_DIR / "config.json"
HISTORY_JSON_PATH = CKPT_DIR / "history.json"
HISTORY_CSV_PATH = CKPT_DIR / "history.csv"
METRICS_PATH = CKPT_DIR / "metrics.json"
PREDICTIONS_PATH = CKPT_DIR / "test_predictions.csv"
CONFUSION_MATRIX_PATH = CKPT_DIR / "confusion_matrix.png"

SEED = int(CFG["seed"])


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_IDS = list(range(torch.cuda.device_count())) if CUDA_AVAILABLE else []
USE_DATA_PARALLEL = len(GPU_IDS) > 1
device = torch.device("cuda:0" if CUDA_AVAILABLE else "cpu")
USE_AMP = bool(CFG["use_amp"] and CUDA_AVAILABLE)

print("WORKDIR:", WORKDIR)
print("RUN_NAME:", RUN_NAME)
print("CKPT_DIR:", CKPT_DIR)
print("device:", device)
print("GPU_IDS:", GPU_IDS)
for gpu_id in GPU_IDS:
    print(f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
print("USE_DATA_PARALLEL:", USE_DATA_PARALLEL)
print("USE_AMP:", USE_AMP)
print("WANDB_AVAILABLE:", WANDB_AVAILABLE)
print("WANDB_MODE:", CFG["wandb_mode"])


In [ ]:
def find_caers_root(workdir):
    workdir = Path(workdir).resolve()
    preferred = workdir / "CAER-S"
    if (preferred / "train").exists() and (preferred / "test").exists():
        return preferred

    candidates = []
    for base in [workdir, workdir / "data", workdir / "datasets"]:
        if not base.exists():
            continue
        for p in base.rglob("*"):
            if p.is_dir() and (p / "train").exists() and (p / "test").exists():
                candidates.append(p)

    if not candidates:
        raise FileNotFoundError(
            "Dataset CAER-S tidak ditemukan. Letakkan dataset di repo sebagai "
            f"{workdir / 'CAER-S'} dengan subfolder train/ dan test/."
        )

    return sorted(candidates, key=lambda x: len(str(x)))[0]


CAERS_ROOT = find_caers_root(WORKDIR)
CAERS_ROOT


In [ ]:
TRAIN_ROOT = CAERS_ROOT / "train"
TEST_ROOT = CAERS_ROOT / "test"

print("CAERS_ROOT:", CAERS_ROOT)
print("TRAIN_ROOT:", TRAIN_ROOT)
print("TEST_ROOT:", TEST_ROOT)
print("Train classes:", sorted(p.name for p in TRAIN_ROOT.iterdir() if p.is_dir()))
print("Test classes:", sorted(p.name for p in TEST_ROOT.iterdir() if p.is_dir()))


In [ ]:
print("Repo files:")
for p in sorted(WORKDIR.iterdir()):
    if p.name in {".git", ".venv", "CAER-S"}:
        continue
    print("-", p.name)


In [ ]:
print("Dataset summary")
for split_dir in [TRAIN_ROOT, TEST_ROOT]:
    counts = {}
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        counts[class_dir.name] = sum(1 for x in class_dir.iterdir() if x.is_file())
    print(split_dir.name, counts)


In [ ]:
DETECTOR_DIR.mkdir(exist_ok=True)

FILES = {
    "train.txt": "https://drive.google.com/file/d/1Em4LUdJ6VS8sOo_XdBi96xu8nwoo_Y4g/view?usp=sharing",
    "val.txt":   "https://drive.google.com/file/d/1thrfz6IdPIXSQRZ6LW-5tGVabbfHUoeA/view?usp=sharing",
    "test.txt":  "https://drive.google.com/file/d/1eEpjAmLrz9f9_SqZh5qCPsScHikg91V4/view?usp=sharing",
    "caers.pth": "https://drive.google.com/file/d/1HxHZQmWnXbhYV0_HU2q-2fcC3twtGJZp/view?usp=sharing",
}

for filename, url in FILES.items():
    out = DETECTOR_DIR / filename
    if not out.exists():
        !gdown --fuzzy "{url}" -O "{out}"

for p in sorted(DETECTOR_DIR.iterdir()):
    print(f"{p.name}: {p.stat().st_size / 1024:.1f} KiB")


# 4. Build manifest bersih dari detector file

In [ ]:
required_detector_files = ["train.txt", "val.txt", "test.txt"]
missing_detector_files = [name for name in required_detector_files if not (DETECTOR_DIR / name).exists()]
if missing_detector_files:
    raise FileNotFoundError(f"Detector files belum lengkap: {missing_detector_files}")

for name in required_detector_files:
    path = DETECTOR_DIR / name
    lines = path.read_text().strip().splitlines()
    print(name, "lines:", len(lines), "sample:", lines[0] if lines else "EMPTY")


In [ ]:
def parse_detector_line(line):
    parts = [p.strip() for p in line.split(",")]
    if len(parts) != 6:
        raise ValueError(f"Invalid detector line: {line}")
    rel_path_raw, raw_label, x1, y1, x2, y2 = parts
    bbox = [int(float(x1)), int(float(y1)), int(float(x2)), int(float(y2))]
    return rel_path_raw, raw_label, bbox


for name in required_detector_files:
    first_line = (DETECTOR_DIR / name).read_text().strip().splitlines()[0]
    print(name, parse_detector_line(first_line))


In [ ]:
print("CAER-S root contents:")
for p in sorted(CAERS_ROOT.iterdir()):
    print(p.name, "DIR" if p.is_dir() else "FILE")

print()
print("Detector directory:")
for p in sorted(DETECTOR_DIR.iterdir()):
    print(p.name)


In [ ]:
from pathlib import Path
import json
import pandas as pd

CLASS_NAMES = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]

CLASS_ALIASES = {
    "Anger": ["Anger", "Angry", "anger", "angry"],
    "Disgust": ["Disgust", "disgust"],
    "Fear": ["Fear", "fear"],
    "Happy": ["Happy", "happy"],
    "Neutral": ["Neutral", "neutral"],
    "Sad": ["Sad", "sad"],
    "Surprise": ["Surprise", "surprise"],
}

LABEL_ID_TO_NAME = {
    0: "Anger",
    1: "Disgust",
    2: "Fear",
    3: "Happy",
    4: "Neutral",
    5: "Sad",
    6: "Surprise",
}


def canonical_label(name):
    name = str(name).strip()

    for canonical, aliases in CLASS_ALIASES.items():
        if name in aliases:
            return canonical

    return name


def infer_label(rel_path_raw, raw_label):
    rel_path_raw = rel_path_raw.replace("\\", "/").strip()
    parts = rel_path_raw.split("/")

    if len(parts) >= 2:
        class_from_path = canonical_label(parts[0])
        if class_from_path in CLASS_NAMES:
            return class_from_path

    raw_label = str(raw_label).strip()

    if raw_label.isdigit():
        idx = int(raw_label)
        if idx in LABEL_ID_TO_NAME:
            return LABEL_ID_TO_NAME[idx]

    return canonical_label(raw_label)


def resolve_class_folder(dataset_root, split_base, canonical):
    """
    Cari nama folder class sebenarnya di dataset.
    Contoh:
    canonical Anger bisa cocok ke folder Anger atau Angry.
    """
    split_dir = Path(dataset_root) / split_base

    if not split_dir.exists():
        raise FileNotFoundError(f"Split folder tidak ada: {split_dir}")

    aliases = CLASS_ALIASES.get(canonical, [canonical])

    for alias in aliases:
        candidate = split_dir / alias
        if candidate.exists():
            return alias

    available = [p.name for p in split_dir.iterdir() if p.is_dir()]
    raise FileNotFoundError(
        f"Tidak menemukan folder untuk class {canonical} di {split_dir}. "
        f"Folder tersedia: {available}"
    )


def resolve_image_path(rel_path_raw, split, dataset_root, raw_label):
    rel_path_raw = rel_path_raw.replace("\\", "/").strip()
    parts = rel_path_raw.split("/")

    # CAER-S official: val tetap berasal dari folder train
    split_base = "test" if split == "test" else "train"

    # Jika detector file sudah mengandung train/test di awal
    if parts[0] in {"train", "test"}:
        split_base = parts[0]
        parts = parts[1:]

    if len(parts) < 2:
        raise ValueError(f"Path detector tidak valid: {rel_path_raw}")

    filename = "/".join(parts[1:])
    label = infer_label(rel_path_raw, raw_label)

    real_class_folder = resolve_class_folder(dataset_root, split_base, label)

    image_path = f"{split_base}/{real_class_folder}/{filename}"
    full_path = Path(dataset_root) / image_path

    if not full_path.exists():
        raise FileNotFoundError(f"Missing image: {full_path}")

    return image_path, label


def parse_detector_file(detector_path, split, dataset_root):
    detector_path = Path(detector_path)
    dataset_root = Path(dataset_root)

    rows = []
    lines = detector_path.read_text().strip().splitlines()

    for idx, line in enumerate(lines):
        parts = [p.strip() for p in line.split(",")]

        if len(parts) != 6:
            raise ValueError(f"Invalid line {idx + 1}: {line}")

        rel_path_raw, raw_label, x1, y1, x2, y2 = parts

        image_path, label = resolve_image_path(
            rel_path_raw=rel_path_raw,
            split=split,
            dataset_root=dataset_root,
            raw_label=raw_label,
        )

        bbox = [int(float(x1)), int(float(y1)), int(float(x2)), int(float(y2))]

        sample_id = f"{split}__{image_path}".replace("/", "__").replace(".", "_")

        rows.append({
            "sample_id": sample_id,
            "image_path": image_path,
            "label": label,
            "split": split,
            "face_bbox": bbox,
        })

    return rows


In [ ]:
train_rows = parse_detector_file(DETECTOR_DIR / "train.txt", "train", CAERS_ROOT)
val_rows   = parse_detector_file(DETECTOR_DIR / "val.txt", "val", CAERS_ROOT)
test_rows  = parse_detector_file(DETECTOR_DIR / "test.txt", "test", CAERS_ROOT)

manifest = train_rows + val_rows + test_rows

print("Total:", len(manifest))
print("Train:", len(train_rows))
print("Val:", len(val_rows))
print("Test:", len(test_rows))


In [ ]:
with MANIFEST_PATH.open("w") as f:
    for row in manifest:
        f.write(json.dumps(row) + "\n")

df = pd.DataFrame(manifest)

print(df["split"].value_counts())
print()
print(pd.crosstab(df["label"], df["split"]))
print()
print("bbox missing:", df["face_bbox"].isna().sum())

print("Saved:", MANIFEST_PATH)


# 5. Dataset class CAER-S two-stream

In [ ]:
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch
import json
from pathlib import Path


def apply_face_mask(image, bbox, fill=(0, 0, 0)):
    masked = image.copy()
    draw = ImageDraw.Draw(masked)

    x1, y1, x2, y2 = [int(v) for v in bbox]
    draw.rectangle([x1, y1, x2, y2], fill=fill)

    return masked


class CAERSTwoStreamDataset(Dataset):
    def __init__(
        self,
        manifest_path,
        dataset_root,
        split,
        face_transform=None,
        context_transform=None,
    ):
        self.manifest_path = Path(manifest_path)
        self.dataset_root = Path(dataset_root)
        self.split = split

        self.face_transform = face_transform
        self.context_transform = context_transform

        self.rows = []

        with self.manifest_path.open("r") as f:
            for line in f:
                row = json.loads(line)
                if row["split"] == split:
                    self.rows.append(row)

        if len(self.rows) == 0:
            raise ValueError(f"Tidak ada data untuk split={split}")

        self.label_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
        self.idx_to_label = {idx: name for name, idx in self.label_to_idx.items()}

    def __len__(self):
        return len(self.rows)

    def crop_face(self, image, bbox):
        x1, y1, x2, y2 = [int(v) for v in bbox]
        return image.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.rows[idx]

        image_path = self.dataset_root / row["image_path"]
        image = Image.open(image_path).convert("RGB")

        bbox = row["face_bbox"]
        label_name = row["label"]

        face_image = self.crop_face(image, bbox)
        context_image = apply_face_mask(image, bbox)

        if self.face_transform is not None:
            face_image = self.face_transform(face_image)

        if self.context_transform is not None:
            context_image = self.context_transform(context_image)

        label = self.label_to_idx[label_name]

        return {
            "face": face_image,
            "context": context_image,
            "label": torch.tensor(label, dtype=torch.long),
            "label_name": label_name,
            "image_path": row["image_path"],
            "bbox": torch.tensor(bbox, dtype=torch.float32),
        }


# 5.1. Transform official-style

In [ ]:
face_train_t = T.Compose([
    T.Resize((96, 96)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

context_train_t = T.Compose([
    T.Resize((128, 171)),
    T.RandomCrop(112),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

face_eval_t = T.Compose([
    T.Resize((96, 96)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

context_eval_t = T.Compose([
    T.Resize((128, 171)),
    T.CenterCrop(112),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])


In [ ]:
if not MANIFEST_PATH.exists():
    required_names = ["parse_detector_file", "DETECTOR_DIR", "CAERS_ROOT"]
    missing_names = [name for name in required_names if name not in globals()]
    if missing_names:
        raise RuntimeError(
            "Manifest belum ada dan fungsi build manifest belum tersedia. "
            f"Jalankan cell setup + build manifest terlebih dahulu. Missing: {missing_names}"
        )

    print("Manifest belum ada. Building:", MANIFEST_PATH)
    train_rows = parse_detector_file(DETECTOR_DIR / "train.txt", "train", CAERS_ROOT)
    val_rows = parse_detector_file(DETECTOR_DIR / "val.txt", "val", CAERS_ROOT)
    test_rows = parse_detector_file(DETECTOR_DIR / "test.txt", "test", CAERS_ROOT)
    manifest = train_rows + val_rows + test_rows

    with MANIFEST_PATH.open("w") as f:
        for row in manifest:
            f.write(json.dumps(row) + "\n")

    df = pd.DataFrame(manifest)
    print(df["split"].value_counts())
    print("Saved:", MANIFEST_PATH)
else:
    print("Using existing manifest:", MANIFEST_PATH)

manifest_rows = []
with MANIFEST_PATH.open("r") as f:
    for line in f:
        manifest_rows.append(json.loads(line))

manifest_df = pd.DataFrame(manifest_rows)
print("Manifest samples:", len(manifest_df))
print(manifest_df["split"].value_counts())
print(pd.crosstab(manifest_df["label"], manifest_df["split"]))

missing_images = []
invalid_bboxes = []
for row in manifest_rows:
    image_path = CAERS_ROOT / row["image_path"]
    if not image_path.exists():
        missing_images.append(row["image_path"])
        continue
    bbox = row.get("face_bbox")
    if not isinstance(bbox, list) or len(bbox) != 4:
        invalid_bboxes.append(row["sample_id"])
        continue
    x1, y1, x2, y2 = [int(v) for v in bbox]
    if x2 <= x1 or y2 <= y1:
        invalid_bboxes.append(row["sample_id"])

print("Missing images:", len(missing_images))
print("Invalid bboxes:", len(invalid_bboxes))
if missing_images[:5]:
    print("Missing examples:", missing_images[:5])
if invalid_bboxes[:5]:
    print("Invalid bbox examples:", invalid_bboxes[:5])
if missing_images or invalid_bboxes:
    raise ValueError("Manifest validation failed. Perbaiki missing image atau invalid bbox sebelum training.")

train_ds = CAERSTwoStreamDataset(
    manifest_path=MANIFEST_PATH,
    dataset_root=CAERS_ROOT,
    split="train",
    face_transform=face_train_t,
    context_transform=context_train_t,
)

val_ds = CAERSTwoStreamDataset(
    manifest_path=MANIFEST_PATH,
    dataset_root=CAERS_ROOT,
    split="val",
    face_transform=face_eval_t,
    context_transform=context_eval_t,
)

test_ds = CAERSTwoStreamDataset(
    manifest_path=MANIFEST_PATH,
    dataset_root=CAERS_ROOT,
    split="test",
    face_transform=face_eval_t,
    context_transform=context_eval_t,
)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))
print("Labels:", train_ds.label_to_idx)


In [ ]:
def denorm(x):
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = x.cpu() * std + mean
    return x.clamp(0, 1)


sample = train_ds[0]

face = denorm(sample["face"]).permute(1, 2, 0)
context = denorm(sample["context"]).permute(1, 2, 0)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(face)
plt.title(f"Face: {sample['label_name']}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(context)
plt.title("Context masked")
plt.axis("off")

plt.show()

print("image_path:", sample["image_path"])
print("bbox:", sample["bbox"])
print("label:", sample["label_name"])


# 7. CAER-Net training baseline

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2, 2) if pool else nn.Identity()

    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.pool(x)
        return x


class CAERCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = ConvBlock(3, 32, pool=True)
        self.conv2 = ConvBlock(32, 64, pool=True)
        self.conv3 = ConvBlock(64, 128, pool=True)
        self.conv4 = ConvBlock(128, 256, pool=True)
        self.conv5 = ConvBlock(256, 256, pool=False)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        return x


class ContextAttention(nn.Module):
    def __init__(self, channels=256):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, 128, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(128)
        self.conv2 = nn.Conv2d(128, 1, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        a = F.relu(self.bn1(self.conv1(x)))
        a = self.conv2(a)
        b, _, h, w = a.shape
        a = F.softmax(a.view(b, -1), dim=1).view(b, 1, h, w)
        return x * a


class AdaptiveFusion(nn.Module):
    def __init__(self, channels=256, num_classes=7, dropout=0.5):
        super().__init__()

        self.face_w1 = nn.Conv2d(channels, 128, kernel_size=1)
        self.face_w2 = nn.Conv2d(128, 1, kernel_size=1)

        self.ctx_w1 = nn.Conv2d(channels, 128, kernel_size=1)
        self.ctx_w2 = nn.Conv2d(128, 1, kernel_size=1)

        self.classifier = nn.Sequential(
            nn.Conv2d(channels * 2, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(128, num_classes, kernel_size=1),
        )

    def forward(self, face_feat, ctx_feat):
        # ------------------------------------------------------------------
        # Important fix:
        # face feature:    [B, 256, 6, 6] from 96x96 input
        # context feature: [B, 256, 7, 7] from 112x112 input
        # We align them before fusion.
        # ------------------------------------------------------------------
        if face_feat.shape[-2:] != ctx_feat.shape[-2:]:
            target_h = min(face_feat.shape[-2], ctx_feat.shape[-2])
            target_w = min(face_feat.shape[-1], ctx_feat.shape[-1])

            face_feat = F.adaptive_avg_pool2d(face_feat, (target_h, target_w))
            ctx_feat = F.adaptive_avg_pool2d(ctx_feat, (target_h, target_w))

        lam_f = self.face_w2(F.relu(self.face_w1(face_feat)))
        lam_c = self.ctx_w2(F.relu(self.ctx_w1(ctx_feat)))

        lam_f = F.adaptive_avg_pool2d(lam_f, 1).flatten(1)
        lam_c = F.adaptive_avg_pool2d(lam_c, 1).flatten(1)

        weights = F.softmax(torch.cat([lam_f, lam_c], dim=1), dim=1)

        wf = weights[:, 0:1, None, None]
        wc = weights[:, 1:2, None, None]

        fused = torch.cat([face_feat * wf, ctx_feat * wc], dim=1)

        logits = self.classifier(fused)
        logits = F.adaptive_avg_pool2d(logits, 1).flatten(1)

        return logits, weights


class CAERNetS(nn.Module):
    def __init__(self, num_classes=7, dropout=0.5):
        super().__init__()
        self.face_encoder = CAERCNN()
        self.context_encoder = CAERCNN()
        self.context_attention = ContextAttention(256)
        self.fusion = AdaptiveFusion(256, num_classes, dropout)

    def forward(self, face, context):
        face_feat = self.face_encoder(face)
        ctx_feat = self.context_encoder(context)
        ctx_feat = self.context_attention(ctx_feat)
        logits, weights = self.fusion(face_feat, ctx_feat)
        return {
            "logits": logits,
            "fusion_weights": weights,
        }


# 8. Training loop

In [ ]:
PER_GPU_BATCH_SIZE = int(CFG["per_gpu_batch_size"])
BATCH_SIZE = PER_GPU_BATCH_SIZE * max(1, len(GPU_IDS))
NUM_WORKERS = min(8, int(CFG["num_workers_per_gpu"]) * max(1, len(GPU_IDS))) if os.name != "nt" else 0
PIN_MEMORY = device.type == "cuda"

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("PER_GPU_BATCH_SIZE:", PER_GPU_BATCH_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)
print("PIN_MEMORY:", PIN_MEMORY)


In [ ]:
def batch_to_device(batch, device):
    return (
        batch["face"].to(device, non_blocking=PIN_MEMORY),
        batch["context"].to(device, non_blocking=PIN_MEMORY),
        batch["label"].to(device, non_blocking=PIN_MEMORY),
    )


def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, use_amp=False, grad_clip_max_norm=None):
    model.train()

    total_loss = 0
    total_correct = 0
    total = 0

    for batch in tqdm(loader, desc="train"):
        face, context, labels = batch_to_device(batch, device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=bool(use_amp)):
            out = model(face, context)
            logits = out["logits"]
            loss = criterion(logits, labels)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            if grad_clip_max_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_max_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip_max_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_max_norm)
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return {
        "loss": total_loss / total,
        "acc": total_correct / total,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device, use_amp=False):
    model.eval()

    total_loss = 0
    total_correct = 0
    total = 0

    all_preds = []
    all_labels = []
    all_paths = []

    for batch in tqdm(loader, desc="eval"):
        face, context, labels = batch_to_device(batch, device)

        with torch.autocast(device_type=device.type, enabled=bool(use_amp)):
            out = model(face, context)
            logits = out["logits"]
            loss = criterion(logits, labels)

        preds = logits.argmax(1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_paths.extend(batch.get("image_path", [""] * labels.size(0)))

    return {
        "loss": total_loss / total,
        "acc": total_correct / total,
        "preds": all_preds,
        "labels": all_labels,
        "image_paths": all_paths,
    }


# 8.1. Training:

In [ ]:
base_model = CAERNetS(num_classes=7, dropout=0.5).to(device)

batch = next(iter(train_loader))
face = batch["face"].to(device, non_blocking=PIN_MEMORY)
context = batch["context"].to(device, non_blocking=PIN_MEMORY)

with torch.no_grad():
    face_feat = base_model.face_encoder(face)
    ctx_feat = base_model.context_encoder(context)
    ctx_feat = base_model.context_attention(ctx_feat)

print("face input:", face.shape)
print("context input:", context.shape)
print("face feat:", face_feat.shape)
print("context feat:", ctx_feat.shape)

out = base_model(face, context)
print("logits:", out["logits"].shape)
print("fusion weights:", out["fusion_weights"].shape)

if USE_DATA_PARALLEL:
    model = nn.DataParallel(base_model, device_ids=GPU_IDS, output_device=GPU_IDS[0])
    print("Training with DataParallel on GPUs:", GPU_IDS)
else:
    model = base_model
    print("Training on single device:", device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=float(CFG["lr"]),
    momentum=float(CFG["momentum"]),
    weight_decay=float(CFG["weight_decay"]),
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=int(CFG["scheduler_step_size"]),
    gamma=float(CFG["scheduler_gamma"]),
)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


## Restart parameter (for retrain loop)

In [ ]:
# import gc
# import torch

# # 1. Hapus variabel utama yang memakan memori GPU
# try:
#     del model
#     del optimizer
#     del scheduler
#     del train_metrics
#     del val_metrics
# except NameError:
#     pass  # Mengantisipasi jika variabel sudah terhapus sebelumnya

# # 2. Paksa Python membersihkan object yang tidak terpakai
# gc.collect()

# # 3. Kosongkan cache memori di kartu grafis (CUDA)
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# print("Memori GPU berhasil dibersihkan!")


In [ ]:
def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def save_history(history, json_path, csv_path):
    with json_path.open("w") as f:
        json.dump(history, f, indent=2)
    if history:
        with csv_path.open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
            writer.writeheader()
            writer.writerows(history)


def checkpoint_payload(epoch, best_val_acc, history):
    return {
        "run_name": RUN_NAME,
        "model_state_dict": unwrap_model(model).state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "epoch": epoch,
        "best_val_acc": best_val_acc,
        "class_names": CLASS_NAMES,
        "history": history,
        "cfg": CFG,
        "gpu_ids": GPU_IDS,
        "data_parallel": USE_DATA_PARALLEL,
        "batch_size": BATCH_SIZE,
        "use_amp": USE_AMP,
    }


def load_training_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    unwrap_model(model).load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if ckpt.get("scaler_state_dict") is not None and scaler is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt


CKPT_DIR.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open("w") as f:
    json.dump(CFG, f, indent=2)

wandb_run = None
if WANDB_AVAILABLE:
    wandb_run = wandb.init(
        project=CFG["project_name"],
        entity=CFG["wandb_entity"] or None,
        name=RUN_NAME,
        mode=CFG["wandb_mode"],
        config={
            **CFG,
            "batch_size": BATCH_SIZE,
            "gpu_ids": GPU_IDS,
            "data_parallel": USE_DATA_PARALLEL,
            "use_amp_actual": USE_AMP,
            "train_samples": len(train_ds),
            "val_samples": len(val_ds),
            "test_samples": len(test_ds),
        },
    )
else:
    print("wandb tidak tersedia. Training tetap berjalan tanpa W&B logging.")

best_val_acc = 0
history = []
start_epoch = 0
patience_counter = 0

resume_from = str(CFG.get("resume_from", "")).strip()
if resume_from:
    ckpt = load_training_checkpoint(resume_from)
    start_epoch = int(ckpt.get("epoch", -1)) + 1
    best_val_acc = float(ckpt.get("best_val_acc", 0))
    history = list(ckpt.get("history", []))
    print(f"Resumed from {resume_from}: start_epoch={start_epoch}, best_val_acc={best_val_acc:.4f}")

EPOCHS = int(CFG["epochs"])
grad_clip_max_norm = CFG.get("grad_clip_max_norm")
grad_clip_max_norm = float(grad_clip_max_norm) if grad_clip_max_norm is not None else None
patience = int(CFG["early_stopping_patience"])
early_stopping_enabled = bool(CFG["early_stopping_enabled"])

for epoch in range(start_epoch, EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
        scaler=scaler,
        use_amp=USE_AMP,
        grad_clip_max_norm=grad_clip_max_norm,
    )
    val_metrics = evaluate(model, val_loader, criterion, device, use_amp=USE_AMP)

    scheduler.step()

    row = {
        "run_name": RUN_NAME,
        "epoch": epoch + 1,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "lr": optimizer.param_groups[0]["lr"],
        "batch_size": BATCH_SIZE,
        "gpu_ids": json.dumps(GPU_IDS),
        "data_parallel": USE_DATA_PARALLEL,
        "use_amp": USE_AMP,
    }

    history.append(row)
    save_history(history, HISTORY_JSON_PATH, HISTORY_CSV_PATH)

    print(row)

    if wandb_run is not None:
        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_metrics["loss"],
            "train/acc": train_metrics["acc"],
            "val/loss": val_metrics["loss"],
            "val/acc": val_metrics["acc"],
            "lr": optimizer.param_groups[0]["lr"],
            "best_val_acc": best_val_acc,
        })

    torch.save(checkpoint_payload(epoch, best_val_acc, history), LAST_CKPT_PATH)
    print("Saved last checkpoint:", LAST_CKPT_PATH)

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        patience_counter = 0
        torch.save(checkpoint_payload(epoch, best_val_acc, history), BEST_CKPT_PATH)
        print("Saved best model:", BEST_CKPT_PATH)
        if wandb_run is not None:
            wandb.run.summary["best_val_acc"] = best_val_acc
    else:
        patience_counter += 1

    print(f"Best val acc: {best_val_acc:.4f} | patience: {patience_counter}/{patience}")

    if early_stopping_enabled and patience_counter >= patience:
        print("Early stopping triggered.")
        break

print("History saved:", HISTORY_JSON_PATH)
print("History CSV saved:", HISTORY_CSV_PATH)
print("Best val acc:", best_val_acc)


# 9. Evaluasi test

In [ ]:
ckpt_path = BEST_CKPT_PATH
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint belum ada: {ckpt_path}. Jalankan cell training terlebih dahulu.")

ckpt = torch.load(ckpt_path, map_location=device)
unwrap_model(model).load_state_dict(ckpt["model_state_dict"])

test_metrics = evaluate(model, test_loader, criterion, device, use_amp=USE_AMP)

y_true = test_metrics["labels"]
y_pred = test_metrics["preds"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    zero_division=0,
)

per_class = {
    CLASS_NAMES[i]: {
        "precision": float(precision[i]),
        "recall": float(recall[i]),
        "f1": float(f1[i]),
        "support": int(support[i]),
    }
    for i in range(len(CLASS_NAMES))
}

metrics_out = {
    "run_name": RUN_NAME,
    "checkpoint": str(ckpt_path),
    "test_loss": float(test_metrics["loss"]),
    "test_acc": float(acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "per_class": per_class,
}

with METRICS_PATH.open("w") as f:
    json.dump(metrics_out, f, indent=2)

with PREDICTIONS_PATH.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image_path", "label", "label_name", "pred", "pred_name", "correct"])
    writer.writeheader()
    for image_path, label, pred in zip(test_metrics["image_paths"], y_true, y_pred):
        writer.writerow({
            "image_path": image_path,
            "label": int(label),
            "label_name": CLASS_NAMES[int(label)],
            "pred": int(pred),
            "pred_name": CLASS_NAMES[int(pred)],
            "correct": int(label) == int(pred),
        })

print("Test Accuracy:", acc)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)
print("Metrics saved:", METRICS_PATH)
print("Predictions saved:", PREDICTIONS_PATH)

print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("CAER-S CAER-Net Confusion Matrix")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=200)
plt.show()
print("Confusion matrix saved:", CONFUSION_MATRIX_PATH)

if globals().get("wandb_run") is not None:
    wandb.log({
        "test/loss": test_metrics["loss"],
        "test/acc": acc,
        "test/macro_f1": macro_f1,
        "test/weighted_f1": weighted_f1,
        "test/confusion_matrix": wandb.Image(str(CONFUSION_MATRIX_PATH)),
    })
    artifact = wandb.Artifact(name=f"{RUN_NAME}-outputs", type="caernet-run")
    for path in [BEST_CKPT_PATH, LAST_CKPT_PATH, CONFIG_PATH, HISTORY_JSON_PATH, HISTORY_CSV_PATH, METRICS_PATH, PREDICTIONS_PATH, CONFUSION_MATRIX_PATH]:
        if path.exists():
            artifact.add_file(str(path))
    wandb.log_artifact(artifact)
    wandb.finish()
